# Chapter 9: The quoted numbers, recomputed

*Code cells: MIT licence (`LICENSE`). Text and figures: CC BY 4.0 (`DATA_LICENSE.md`). Part of the code-and-data repository for "Mean span dimension of an ideal wormlike chain" (Y. Wang, 2026).*

In [1]:
import sys, json, math, time
from pathlib import Path
ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'tutorial' else Path.cwd().resolve()
sys.path.insert(0, str(ROOT / 'code'))
import numpy as np, pandas as pd, sympy as sp
import matplotlib.pyplot as plt
pd.set_option('display.precision', 8)
plt.rcParams.update({'figure.dpi': 110, 'font.size': 9})
REF = pd.read_csv(ROOT / 'data' / 'wlc_mean_span_reference.csv')
LAM, F = REF['lambda'].to_numpy(), REF['F_continuum'].to_numpy()
Z_M, B_COEF, A_GAUSS = 0.71821164, 0.64680, 4/math.sqrt(3*math.pi)
def check(name, value, expected, tol):
    ok = abs(value - expected) <= tol
    print(('PASS' if ok else 'FAIL') + f'  {name}: {value:.10g}  (expected {expected:.10g}, tol {tol:g})')
    return ok

This chapter recomputes the quantitative statements of the article from the archived data and prints
a pass/fail line for each. Two kinds of test are used: `rec` checks a quoted value to the rounding of
the printed number; `upto` checks a stated bound one-sidedly ("below", "within", "at most"). The list
covers the numerical solution, the results and the numerical methods of the supplementary material.

In [2]:
import plot_main_figures as pmf
from scipy.optimize import brentq
results = []
def rec(name, value, expected, tol):
    results.append(check(name, value, expected, tol))
def upto(name, value, bound):
    ok = value <= bound
    print(('PASS' if ok else 'FAIL') + f'  {name}: {value:.4g}  (bound {bound:g})')
    results.append(ok)
Fi = lambda x: float(F[np.isclose(LAM, x)][0])
coil = 4/math.sqrt(math.pi); Rg = np.sqrt(pmf.rg2_over_p2(LAM)); ratio = F/Rg
short = pmf.short_expansion(LAM); longc = pmf.long_expansion(LAM)
e_short = np.abs(short/F-1); e_long = np.abs(longc/F-1)
# Numerical solution: the table and its accuracy
rec('number of nodes', len(LAM), 67, 0)
rec('max internal relative sensitivity (1.9e-5)', REF.epsilon_internal_rel.max(), 1.86e-5, 5e-7)
rec('max internal absolute sensitivity (1.5e-5, at 7.5e4)', REF.epsilon_internal_abs.max(), 1.5e-5, 5e-7)
upto('all eps_int components below 4.2e-8 for lambda>=5000', REF.epsilon_internal_rel[LAM>=5000].max(), 4.2e-8)
upto('order dependence |F96-Finf|/F below 6e-8 for lambda>1e3', (np.abs(REF.F_m96-REF.F_continuum)/REF.F_continuum)[LAM>1e3].max(), 6e-8)
spot = pd.read_csv(ROOT/'data'/'high_order_spot_checks.csv').set_index('lambda')
for lam, q in ((0.1, 5e-9), (1.0, 3e-9), (10.0, 9e-10), (100.0, 4e-11), (1000.0, 2e-9)):
    rec(f'high-order extrapolation vs table at {lam:g} ({q:g})', abs(spot.loc[lam, 'relative_difference_extrapolation_pct'])/100, q, 0.5*q)
rec('high-order extrapolation vs table at 0.01 (1.4e-6)', spot.loc[0.01, 'relative_difference_extrapolation_pct']/100, 1.4e-6, 5e-8)
sc = spot[['scaled_residual_m128', 'scaled_residual_m192', 'scaled_residual_m256']]
upto('(F_m-F_inf) m^3 constant within 2.5 % for 0.1<=lambda<=100', float(((sc.max(axis=1)-sc.min(axis=1))/sc.max(axis=1)).loc[[0.1, 1.0, 10.0, 100.0]].max()), 0.025)
stiff = pd.read_csv(ROOT/'data'/'stiff_alias_summary.csv').set_index('lambda')
for lam, q in ((0.0125, 7.8e-7), (0.025, 9e-8)):
    rec(f'table vs m<=512 diagnostics at {lam:g} ({q:g})', (stiff.loc[lam, 'F_extrap']-Fi(lam))/stiff.loc[lam, 'F_extrap'], q, 0.05*q)
upto('table vs m<=512 diagnostics, lambda>=0.05 (< 1e-8)', max(abs(Fi(l)/stiff.loc[l, 'F_extrap']-1) for l in stiff.index if l >= 0.05), 1e-8)
cond = pd.read_csv(ROOT/'data'/'conditioning_audit_full.csv')
upto('cond(E+ Z-) at every production point (below 1.47)', cond.condition_max.max(), 1.47)
rec('stable dimension = m/2 everywhere', float((cond.stable_dimension_min == cond.m/2).all() and (cond.stable_dimension_max == cond.m/2).all()), 1, 0)
reg = pd.read_csv(ROOT/'data'/'solver_regression_audit.csv')
upto('eig vs Schur transform (< 6e-12)', reg.rel_eig_schur.max(), 6e-12)
ver = pd.read_csv(ROOT/'validation'/'reference_table_versions.csv')
upto('Schur vs eig table, former A=22 inversion (<= 1.3e-10)', ver.rel_schur_A22_minus_eig_A22.abs().max(), 1.35e-10)
# The crossover
rec('crossing point 64/(3 pi)', 64/(3*math.pi), 6.79, 0.005)
rec('F at crossing (2.20)', float(np.interp(64/(3*math.pi), LAM, F)), 2.20, 0.005)
rec('overestimate at crossing (54 %)', 100*((64/(3*math.pi))/2/np.interp(64/(3*math.pi), LAM, F)-1), 54, 0.5)
rec('short-chain error at lambda=0.3 (0.10 %)', 100*e_short[LAM==0.3][0], 0.10, 0.005)
upto('short-chain error within 0.3 % for lambda<=0.5', 100*e_short[LAM<=0.5].max(), 0.3)
upto('long-chain error within 0.3 % for lambda>=5', 100*e_long[LAM>=5].max(), 0.3)
rec('long-chain error at 10 (0.07 %)', 100*e_long[LAM==10][0], 0.07, 0.005)
rec('long-chain error at 100 (0.0006 %)', 100*e_long[LAM==100][0], 0.0006, 0.00005)
upto('long-chain error within 4e-8 for lambda>=5000', e_long[LAM>=5000].max(), 4e-8)
rec('Gaussian law overestimate at 100 (12 %)', 100*(pmf.A_GAUSS*10/Fi(100)-1), 12, 0.5)
rec('Gaussian law overestimate at 5000 (1.6 %)', 100*(pmf.A_GAUSS*math.sqrt(5000)/Fi(5000)-1), 1.6, 0.05)
# Span and radius of gyration
rec('X/Rg at 5000 (2.2225)', float(ratio[LAM==5000][0]), 2.2225, 5e-5)
rec('X/Rg at 1e5 (2.2489)', float(ratio[LAM==1e5][0]), 2.2489, 5e-5)
rec('below coil at 5000 (1.5 %)', 100*(1-float(ratio[LAM==5000][0])/coil), 1.5, 0.05)
rec('below coil at 1e5 (0.35 %)', 100*(1-float(ratio[LAM==1e5][0])/coil), 0.35, 0.005)
c1 = math.sqrt(3)*pmf.B_COEF + 6/math.sqrt(math.pi)
ratio_long = lambda l: coil - 2*math.sqrt(3)*Z_M/math.sqrt(l) + c1/l
rec('1 % approach from the long-chain expansion of X/R_g (1.2e4)', brentq(lambda l: 1 - ratio_long(l)/coil - 0.01, 1e3, 1e6), 1.2e4, 500)
# Independent tests, Fig. S1
paths = pd.read_csv(ROOT/'data'/'heat_path_coupled.csv'); fine = paths.sort_values('steps').groupby('lam').tail(1).set_index('lam')
upto('max paired step difference, L/p<=100 (< 0.03 %)', 100*np.abs(paths.pairwise_minus_fine/paths.lam.map(REF.set_index('lambda').F_continuum)).max(), 0.03)
p01 = paths[paths.lam == 0.1]
upto('paired step differences at L/p=0.1 (< 0.002 %)', 100*np.abs(p01.pairwise_minus_fine).max()/float(REF.loc[REF['lambda']==0.1,'F_continuum'].iloc[0]), 0.002)
fv = pd.read_csv(ROOT/'data'/'finite_volume_comparison.csv').set_index('lambda')
for lam, q in ((0.1, 0.011), (1.0, 0.005), (10.0, 0.018)):
    rec(f'finite volume, integer orders, at {lam:g} ({q} %)', fv.loc[lam, 'relative_difference_pct'], q, 0.0006)
rec('finite volume residual / sensitivity at 0.1 (2.0)', fv.loc[0.1, 'relative_difference_pct']/fv.loc[0.1, 'FV_combined_sensitivity_rel_pct'], 2.0, 0.05)
rec('finite volume residual / sensitivity at 10 (1.5)', fv.loc[10.0, 'relative_difference_pct']/fv.loc[10.0, 'FV_combined_sensitivity_rel_pct'], 1.5, 0.05)
fvo = pd.read_csv(ROOT/'data'/'finite_volume_order_analysis.csv').set_index('lambda')
upto('finite volume, observed orders, |residual| (< 4e-5)', fvo.relative_difference_pct.abs().max()/100, 4e-5)
rec('observed order at lambda=1, smallest (0.77)', fvo.loc[1.0, 'observed_order_Ny_min'], 0.77, 0.005)
rec('observed order at lambda=10, largest (0.94)', fvo.loc[10.0, 'observed_order_Ny_max'], 0.94, 0.006)
# The practical formula
err = 100*np.abs(pmf.rational(LAM)/F-1)
split = pd.read_csv(ROOT/'data'/'reference_nodes_with_split.csv').set_index('lambda')['set']; sets = pd.Series(LAM).map(split)
rec('calibration / holdout / extrapolation counts (34/24/9)', 10000*(sets=='calibration').sum()+100*(sets=='holdout').sum()+(sets=='extrapolation').sum(), 342409, 0)
rec('fit error on calibration nodes (0.089 %)', err[(sets=='calibration').values].max(), 0.0887, 0.0005)
rec('fit error on withheld nodes (0.092 %)', err[(sets!='calibration').values].max(), 0.0924, 0.0005)
upto('fit error on nodes > 5000 (< 1.5e-6)', err[LAM>5000].max()/100, 1.5e-6)
# Milne length and B from archived moments
bm = json.load(open(ROOT/'data'/'boundary_moment_summary.json'))
rec('B from boundary moments at m=512 (0.6468)', bm['highest_order']['B_from_moments'], 0.6468, 2e-5)
rec('z_M at m=512 (0.71821164)', bm['highest_order']['z_M'], 0.71821164, 2e-8)
mm = pd.read_csv(ROOT/'data'/'milne_boundary_moments.csv').set_index('m')
for m, b in ((32, 0.646722), (64, 0.646791), (128, 0.646799), (192, 0.646800)):
    rec(f'B(m={m}) ({b})', mm.loc[m, 'B_from_moments'], b, 5e-7)
# Second-round additions: every remaining revised number
ex = pd.read_csv(ROOT/'data'/'endpoint_excursion_diagnostic.csv'); ex = ex[ex['lambda'] <= 0.1]
fin = ex.sort_values('steps').groupby('lambda').tail(1).set_index('lambda')
dev = 100*(fin.excursion_over_lambda2/(9/256)-1); z = (fin.excursion_over_lambda2-9/256)/fin.excursion_se
for lam, q in ((0.005, -0.60), (0.01, 0.10), (0.02, -0.81), (0.05, -0.65), (0.1, -1.28)):
    rec(f'Fig. 3(b) deviation from 9/256 at {lam:g} ({q} %)', dev.loc[lam], q, 0.006)
upto('Fig. 3(b): all points within two standard errors', float(np.abs(z).max()), 2.0)
rec('Gaussian value 4Rg/sqrt(pi) above X at 100 (10 %)', 100*(coil*float(Rg[LAM == 100][0])/Fi(100)-1), 10, 0.5)
rec('noise amplification ratio e^(22/2)/e^(12/2) (e^5 ~ 150)', math.exp(5), 148.4, 0.1)
rec('formula error at lambda=6000, printed coefficients (1.46e-6)', err[LAM == 6000][0]/100, 1.46e-6, 0.01e-6)
ratios = {l: REF.epsilon_internal_rel[np.isclose(LAM, l)].values[0]/abs(spot.loc[l, 'relative_difference_extrapolation_pct']/100) for l in (0.01, 1000.0)}
rec('eps_int / accuracy at 0.01 (14)', ratios[0.01], 14, 0.5)
rec('eps_int / accuracy at 1000 (35)', ratios[1000.0], 35, 1)
nz = pd.read_csv(ROOT/'validation'/'inversion_noise_audit.csv').set_index('lambda')
upto('former A=22 table vs the long-chain expansion, lambda>=5000 (< 1e-6)', nz.dev_longchain_schur_A22[nz.index >= 5000].abs().max(), 1e-6)
upto('current table vs the long-chain expansion, lambda>=5000 (< 4e-8)', nz.dev_longchain_current[nz.index >= 5000].abs().max(), 4e-8)
# Third-round additions
rec('former A=22 table vs the long-chain expansion, lambda>=5000: largest (9e-7)', nz.dev_longchain_schur_A22[nz.index >= 5000].abs().max(), 9.3e-7, 0.1e-7)
tv = pd.read_csv(ROOT/'validation'/'reference_table_versions.csv').set_index('lambda')
rec('A=22 inversion noise at lambda=1000 (a few 1e-8)', abs(tv.loc[1000.0, 'rel_current_minus_schur_A22']), 3.5e-8, 1.5e-8)
tel = pd.read_csv(ROOT/'validation'/'telegraph_inversion_test.csv')
rec('telegraph test: most negative error (-1.5e-10)', tel.rel_error_analytic_transform.min(), -1.5e-10, 0.05e-10)
rec('telegraph test: least negative error (-3e-11)', tel.rel_error_analytic_transform.max(), -2.9e-11, 0.15e-11)
upto('telegraph test: error equals predicted alias (1e-14)', (tel.rel_error_analytic_transform - tel.predicted_rel_residual_alias).abs().max(), 1.5e-14)
upto('telegraph test: solver m=2 vs analytic (7e-12)', (tel.rel_error_solver_m2 - tel.rel_error_analytic_transform).abs().max(), 7e-12)
upto('remaining alias bound 4 e^(-2A) at A=12 (1.6e-10)', 4*math.exp(-24), 1.6e-10)
lc = json.load(open(ROOT/'data'/'long_chain_remainder_check.json'))
lc_dev100 = float(pd.read_csv(ROOT/'data'/'long_chain_remainder_check.csv').set_index('lambda').loc[100.0, 'rel_dev_from_longchain'])
upto('the long-chain expansion + remainder, 1e3<=lambda<=2e4 (2.7e-9)', lc['max_abs_rel_residual_1e3_to_2e4'], 2.7e-9)
upto('  same, as a fraction of eps_int (0.13)', lc['max_abs_rel_residual_over_eps_1e3_to_2e4'], 0.13)
upto('  same with a lambda^-1 or lambda^-1/2 term (< 3e-9)', max(lc['max_abs_rel_residual_1e3_to_2e4_with_lambda_m1'], lc['max_abs_rel_residual_1e3_to_2e4_with_lambda_m1_2']), 3e-9)
upto('  shift of B from the lambda^-1/2 refit (1e-6)', abs(lc['refit_B_shift']), 1.5e-6)
upto('  same with lambda^-2 in place of lambda^-5/2 (< 3e-9)', lc['max_abs_rel_residual_1e3_to_2e4_with_lambda_m2_instead'], 3e-9)
upto('  same with the fit on 100..700 only (2.7e-9)', lc['max_abs_rel_residual_1e3_to_2e4_fit_100_to_700'], 2.7e-9)
rec('deviation from the long-chain expansion at L/p=100 (5.5e-6)', abs(lc_dev100), 5.5e-6, 0.05e-6)
rec('lambda^-3/2 coefficient of the remainder (-0.064)', lc['c2_lambda_m3_2'], -0.064, 0.001)
rec('residual at 7.5e4 (4e-8, 0.96 eps_int)', lc['residual_over_eps_above_2e4']['75000.0'], 0.96, 0.01)
rec('A = 10, 14 change of F_inf for lambda>=1e4 (up to 1.4 eps_int)', lc['max_abs_A_change_over_eps_lambda_ge_1e4'], 1.4, 0.05)
upto('FV: extrapolation-model sensitivity (up to 4e-5)', fvo.spatial_model_change_pct.abs().max()/100, 4.1e-5)
upto('FV: angular-window sensitivity (< 5e-6)', fvo.angular_fit_change_pct.abs().max()/100, 5e-6)
for lam, q in ((0.1, 0.0025), (1.0, 0.0009), (10.0, 0.0003)):
    rec(f'FV, order-trend variant at {lam:g} (+{q} %)', fvo.loc[lam, 'relative_difference_order_trend_pct'], q, 0.00006)
# Long-chain paths (supplementary material)
pl1 = pd.read_csv(ROOT/'data'/'heat_path_long.csv'); pool = pd.read_csv(ROOT/'data'/'heat_path_long_pooled.csv')
pl = pd.concat([pl1[pl1.lam < 5e3], pool])            # L/p = 1e4: four pooled batches of 20 000 chains
v = pool.sort_values('steps').iloc[-1]
rec('pooled path estimate at L/p=1e4 (128.890)', v.span, 128.890, 0.0005)
rec('  its standard error (0.079)', v.span_se, 0.079, 0.0005)
for bm, q in zip([float(x) for x in v.batch_means.split()], (129.09, 128.56, 129.13, 128.77)):
    rec(f'  batch mean ({q})', bm, q, 0.005)
b1 = pl1[pl1.lam == 1e4].sort_values('steps').iloc[-1]
rec('  one batch: standard error (0.16)', b1.span_se, 0.16, 0.005)
pair = {lam: pl[(pl.lam == lam)].sort_values('steps').iloc[-2] for lam in (1000.0, 10000.0)}
rec('paired difference, two finest grids, L/p=1e3 (0.03 %)', 100*pair[1000.0].pairwise_minus_fine/Fi(1000), 0.03, 0.005)
rec('paired difference, two finest grids, L/p=1e4 (0.06 %)', 100*pair[10000.0].pairwise_minus_fine/Fi(10000), 0.06, 0.01)
upto('finest-step bias h^2/24 at L/p=1e4 (< 3e-4)', (10000/131072)**2/24, 3e-4)
sec = [100*float(r.span_se)/Fi(float(r.lam)) for r in pd.concat([paths, pl]).sort_values('steps').groupby('lam').tail(1).itertuples()]
rec('sampling SE, smallest (4e-4)', min(sec)/100, 4.2e-4, 0.3e-4)
rec('sampling SE, largest (9e-4)', max(sec)/100, 9.1e-4, 0.3e-4)
allp = pd.concat([paths, pl]).sort_values('steps').groupby('lam').tail(1)
upto('all six lengths: |path - table| / SE (within 1)', max(abs(float(r.span) - Fi(float(r.lam)))/float(r.span_se) for r in allp.itertuples()), 1.0)
fvl = json.load(open(ROOT/'data'/'finite_volume_long_summary.json'))
rec('FV at L/p=100, observed order (-0.0031 %)', 100*fvl['rel_diff_observed_order'], -0.0031, 0.00006)
rec('FV at L/p=100, integer order (-0.0021 %)', 100*fvl['rel_diff_integer_order'], -0.0021, 0.00006)
rec('FV at L/p=100: observed spatial order (0.98)', fvl['observed_spatial_orders'][-1], 0.98, 0.006)
upto('FV at L/p=100: additivity defect (9e-6)', abs(fvl['additivity_defect_rel']), 9.5e-6)
rec('long-chain offset / F at L/p=100 (12 %)', 100*2*Z_M/Fi(100), 12, 0.5)
# The long-chain offset and the depletion layer
rec('long-chain offset 2 z_M (1.436)', 2*Z_M, 1.436, 0.0005)
rec('wall offset z_M (0.718)', Z_M, 0.718, 0.0005)
g = lambda l: l/3 - 1 + 2/l - 2*(1-math.exp(-l))/l**2
for lam in (1e3, 1e4, 1e5):           # delta = X/2 = 2 Rg/sqrt(pi) - z_M p + O(lambda^-1/2), the wall-offset expansion
    rem = float(F[LAM==lam][0])/2 - (2*math.sqrt(g(lam))/math.sqrt(math.pi) - Z_M)
    rec(f'the wall-offset expansion: remainder x sqrt(L/p) at L/p={lam:g} -> B/2 + sqrt(3/pi) (1.3006)', rem*math.sqrt(lam), B_COEF/2 + math.sqrt(3/math.pi), 0.005)
wall = lambda l: 2*math.sqrt(g(l))/math.sqrt(math.pi) - Fi(l)/2
for lam, q in ((100.0, 0.59), (1000.0, 0.68), (10000.0, 0.71)):
    rec(f'wall offset at L/p={lam:g} ({q} p)', wall(lam), q, 0.005)
rec('short-chain depletion: (L/4 - delta)/(L^2/p) -> 37/1536 at 0.01', (0.01/4 - float(F[LAM==0.01][0])/2)/0.01**2, 37/1536, 2e-4)
print(f'\n{sum(results)} of {len(results)} checks passed')

PASS  number of nodes: 67  (expected 67, tol 0)
PASS  max internal relative sensitivity (1.9e-5): 1.862857525e-05  (expected 1.86e-05, tol 5e-07)
PASS  max internal absolute sensitivity (1.5e-5, at 7.5e4): 1.481043734e-05  (expected 1.5e-05, tol 5e-07)
PASS  all eps_int components below 4.2e-8 for lambda>=5000: 4.167e-08  (bound 4.2e-08)
PASS  order dependence |F96-Finf|/F below 6e-8 for lambda>1e3: 5.531e-08  (bound 6e-08)
PASS  high-order extrapolation vs table at 0.1 (5e-09): 5.386602164e-09  (expected 5e-09, tol 2.5e-09)
PASS  high-order extrapolation vs table at 1 (3e-09): 2.936850652e-09  (expected 3e-09, tol 1.5e-09)
PASS  high-order extrapolation vs table at 10 (9e-10): 8.740349555e-10  (expected 9e-10, tol 4.5e-10)
PASS  high-order extrapolation vs table at 100 (4e-11): 3.778999336e-11  (expected 4e-11, tol 2e-11)
PASS  high-order extrapolation vs table at 1000 (2e-09): 1.92800198e-09  (expected 2e-09, tol 1e-09)
PASS  high-order extrapolation vs table at 0.01 (1.4e-6): 1.3773